In [14]:
from pathlib import Path
import sys
sys.path.append( str( Path("../../../.." ).resolve()) )

# Class: seqAlign 

In [4]:
# Load modules
from pathlib import Path
import sys
sys.path.insert(0,  str(Path("../../../..").resolve()) )  # Add the root directory to the path

import parasail

In [5]:
# Load Test data
ref = "GKGDPKKPRGKMSSYAFFVQTSREEHKKKHPDASVNFSEFSKKCSERWKTMSAKEKGKFEDMAKADKARYEREMKTYIPPKGE"
seq2 = "MVKVLGAGGQIAANRVKVEFKNDPSISELASLVKSQKVTVRGGNKMELQFRGKGYVGEVKEGEEVEGKNKPEEKHD"
seq3 = ref[::-1] # Reverse of ref
seq4 = "PRGKMSSYAFFVQTSREEHKKKHPDASVNFSEFSK"
seq4 = "PRGKMSSYAFFV121QTSREEHK2312KKHPD12321ASVNFSEFSK" # Introduced gaps
seq5 = "KKPRGKMSSYAFFVQTSREEHKKKHPDASVNFSEF"
seq5 = "KKPRGKMSSYAFHPDASVNFSEFFVQTSREEHKKKHPDASVNFSEF" # Introduced repetition



In [6]:
import parasail
class seqAlign():
    def __init__( self, refseq: str, queryseq: str, sanityCheck: bool = True): 
        self.refseq = refseq
        self.queryseq = queryseq
        self.result = None                                                       # parasail alignment result object
        self.reverseQuery = False
        self.sanityCheck = sanityCheck
        self.matched_indices = None                                              # Dictionary of matched indices and AAs {"Ref":{"Seq_Idx":None, "Seq_AA":None }, "Query":{"Seq_Idx":None, "Seq_AA":None }}
        self.matched_indices_map = None                                          # Dictionary mapping matched indices between ref and query sequences {"Ref_to_Query":{ref_idx:query_idx}, "Query_to_Ref":{query_idx:ref_idx}}
        if sanityCheck:                                                                                                 
            self.find_orientation()
            if self.reverseQuery: 
                print("Warning: Query sequence reversed for better alignment")

    def align( self, mode: str = "global", gap_open: int = 10, gap_extend: int = 1,
              matrix: str = "blosum62"):
        if mode == "global":
            self.result = parasail.nw_trace_striped_32( self.queryseq,
                self.refseq, gap_open, gap_extend, getattr( parasail, matrix) )
        elif mode == "local":
            self.result = parasail.sw_trace_striped_32( self.queryseq, 
                self.refseq, gap_open, gap_extend, getattr( parasail, matrix))
        else:
            raise ValueError("Invalid mode. Choose 'global' or 'local'.")
        
        return self
    
    def find_orientation( self, mode: str = "global", gap_open: int = 10, 
                                gap_extend: int = 1, matrix: str = "blosum62"):
        """
        Find if the best alignment is obtained by reversing the query sequence.
        """
        normalScore = self.align(mode = mode, gap_open = gap_open, 
                    gap_extend = gap_extend,matrix = matrix).result.score
        self.queryseq = self.queryseq[::-1]
        reverseScore = self.align(mode = mode, gap_open = gap_open, 
                    gap_extend = gap_extend,matrix = matrix).result.score
        if reverseScore > normalScore:
            self.reverseQuery = True
        else:
            self.queryseq = self.queryseq[::-1]                                 # Restore original query sequence
            self.align(mode = mode, gap_open = gap_open, 
                    gap_extend = gap_extend,matrix = matrix)
            self.reverseQuery = False                                   
        return self
    
    def map_matching_res( self, match_type = "exact"):
        
        """ 
        Map the matched residues between reference and query sequences based
        on alignment result.
        Args:
        - match_type (str ("exact","all") ): Type of match to consider. 
            Options:
            - "exact":  for exact matches only, or 
            - "all" for matched including conservative and semi-conservative matches.
            - "wgaps": include gaps (not implemented yet)
        Returns:
            self: Updated seqAlign object with matched indices.
        """

        if self.result is None: self.align()                                    # Perform alignment if not done already to obtain traceback / self.result 
        matches = {"Ref":{"Seq_Idx":None, "Seq_AA":None },
                   "Query":{"Seq_Idx":None, "Seq_AA":None }}

        if match_type == "exact":matchpattern = ["|"]                           # Exact matches only
        elif match_type == "all": matchpattern = ["|", ":", "."]                # Include conservative and semi-conservative matches
        else:
            raise ValueError("Invalid match_type. Choose 'exact' or 'all'.")                  

        comp_matchIndex=[idx for idx, char in enumerate(self.result.traceback.comp) # Get indices of matches in comparison string 
                           if char in matchpattern ]
   
        def seq_match_idx( seq: str, matchIdx: list[int]) -> list[int]:         # Helper function to get indices
            gapCount = 0
            seq_indices = []
            for seq_idx, char in enumerate(seq):
                if char == '-':
                    gapCount += 1
                if seq_idx in matchIdx:
                    seq_indices.append(seq_idx - gapCount)
            return seq_indices
        
        refSeq_matchIdx = seq_match_idx( self.result.traceback.ref,comp_matchIndex)
        matches["Ref"]["Seq_Idx"] = refSeq_matchIdx 

        queryseq_matchIdx = seq_match_idx( self.result.traceback.query,comp_matchIndex)
        matches["Query"]["Seq_Idx"] = queryseq_matchIdx

        refSeq_matchAA = [ self.refseq[idx] for idx in refSeq_matchIdx ]
        matches["Ref"]["Seq_AA"] = refSeq_matchAA 

        queryseq_matchAA = [ self.queryseq[idx] for idx in queryseq_matchIdx ]
        matches["Query"]["Seq_AA"] = queryseq_matchAA 
        
        self.matched_indices = matches
        self.matched_indices_map = dict(                                        # Map ref seq idx to query seq idx based on alignment
                                zip( self.matched_indices["Ref"]["Seq_Idx"],
                                    self.matched_indices["Query"]["Seq_Idx"]) )                   
        return self
    
    def visualize( self):
        if self.result is None: self.align()                                    # Perform alignment if not done already to obtain traceback / self.result
        comp_size = len(self.result.traceback.comp)
        count = list( " "*comp_size )
        for idx in range(comp_size):
            if idx % 100 - 99 == 0 and idx != 0: count[idx] = "+"
            elif idx % 50 - 49 == 0 and idx != 0: count[idx] = "*"
            elif idx % 10 - 9 == 0 and idx != 0: count[idx] = "|"
        count = "".join(count)
        
        print("      ", count)
        print("Ref:  ", self.result.traceback.ref)
        print("      ", self.result.traceback.comp)
        print("Query:", self.result.traceback.query)
        return self

# Class: model_seqAlign 

In [7]:
import gemmi

from xaidar.data.molecModels import flatten_pdb, get_chain_seq, seqAlign
# from xaidar.data.molecModels import seqAlign
class model_seqAlign( seqAlign):
    def __init__( self, refmodel: gemmi.Structure, querymodel: gemmi.Structure, sanityCheck: bool = True): 
        self.refmodel = refmodel
        self.querymodel = querymodel
        self.refseq = get_chain_seq( refmodel )[0]                              # Get sequence of first chain in REFERENCE model
        self.queryseq = get_chain_seq( querymodel )[0]                          # Get sequence of first chain in QUERY model            
        super().__init__( refseq = self.refseq, queryseq = self.queryseq, 
                                                    sanityCheck = sanityCheck)
        self.ref_res_lst = flatten_pdb( refmodel, "chain")[0].whole()           # List of all residue objects in REFERENCE model
        self.query_res_lst = flatten_pdb( querymodel, "chain")[0].whole()      # List of all residue objects in QUERY model       
        self.matched_indices = None                                             # Dictionary of matched indices and AAs {"Ref":{"Seq_Idx":None, "Seq_AA":None }, "Query":{"Seq_Idx":None, "Seq_AA":None }}
        self.matched_res = { "Ref": None, "Query": None }                       # Dictionary of matched residues {"Ref": [residue objects], "Query": [residue objects]    }
        self.match_status = None                                                # Status of match check (None if not checked, True if all matched, False if mismatch)          

    def map_matching_res(self, match_type = "exact", gaps = False):
        """
        Map matched residues between reference and query models based on sequence alignment.
        Args:
        - match_type (str, optional): Type of match to consider. Defaults to "exact".
            Options:
            - "exact": Only exact matches of amino acids.
            - "all": Includes partial matches and conserved substitutions.
        - gaps (bool, optional): Whether to include gaps in the mapping. Defaults to False.
        Returns:
        - self: Updated object with matched residues.
        """
        super().map_matching_res( match_type = match_type)                      # Call parent method to get matched indices (self.matched_indices)
        self.ref_res_lst  = flatten_pdb(self.refmodel, level = "residue")                 # List of all residue objects in REFERENCE model
        self.query_res_lst = flatten_pdb(self.querymodel, level = "residue")             # List of all residue objects in QUERY model


        if gaps:                                               # Some positions in ref seq may not have a matching position in query seq due to gaps in alignment             
            self.matched_res["Ref"] = self.ref_res_lst
            self.matched_res["Query"] = []                                                            
            for ref_aa_pos in range( len(self.refseq)  ):                            # Loop over all positions in ref seq
                if ref_aa_pos in list( self.matched_indices_map.keys() ):               # If ref position has a matching position in current query seq
                    matching_query_aa_id = self.matched_indices_map[ref_aa_pos]                   # Get matching query seq idx
                    res = self.query_res_lst[ matching_query_aa_id ]                    # Get matching residue object in QUERY model
                    self.matched_res["Query"].append(res)                               # Append matched residue object to list
                else: self.matched_res["Query"].append( None )                      # If ref position has NO matching position in current query seq add None to list
                    
        else:
            self.matched_res["Ref"] = [ self.ref_res_lst[idx] for idx in           # List of matched residue objects in REFERENCE model
                                self.matched_indices["Ref"]["Seq_Idx"]]
            self.matched_res["Query"] = [ self.query_res_lst[idx] for idx in       # List of matched residue objects in QUERY model
                                    self.matched_indices["Query"]["Seq_Idx"] ]
        
        return self


    def check_match(self, verbose = True):
        """
        Check if the matched residues between reference and query models have 
        the same residue serial numbers.
        Args:
        - verbose (bool, optional): Whether to print mismatch information. 
            Defaults to True.
        Returns:
        - self: Updated object with .match_status.
        """
        if self.matched_indices is None:
            self.map_matching_res()
        if len( self.matched_indices["Ref"]["Seq_Idx"]) != len(
                                     self.matched_indices["Query"]["Seq_Idx"]):
            raise ValueError("Mismatch in number of matched amino acids " \
                                                "between reference and query.")
        ref_match_res_ids = [ res.seqid.num for res in 
                                                    self.matched_res["Ref"] ]   # List of matched residue serial numbers in REFERENCE model    
        query_match_res_ids = [ res.seqid.num for res in 
                                                    self.matched_res["Query"]]  # List of matched residue serial numbers in QUERY model
        
        if ref_match_res_ids != query_match_res_ids:
            if verbose:
                print( "Mismatch in residue serial numbers between " \
                                                        "reference and query.")
            self.match_status = False
        else:
            if verbose:
                print("Match of all residue serial numbers between " \
                                                       "reference and query.")
            self.match_status = True
        return self


- ### Test map_matching_res()

In [8]:
# load test pdbs

import gemmi
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
cox2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/coxb4_2a.pdb")
ev2a1_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a_1.pdb")
from xaidar.data.protocols import protein_processing
from xaidar.data.molecModels import sele_pdb, sele_AA, sele_model
from xaidar.data.molecModels import get_pdb_stats
ev2a_prot = protein_processing(ev2a_pdb)
ev2a1_prot = protein_processing(ev2a1_pdb)
cox2a_prot = cox2a_pdb
for foo in [ sele_AA, sele_model]:
    cox2a_prot = sele_pdb(cox2a_prot, foo)

In [12]:
from xaidar.data.molecModels import sele_pdb, sele_res_idx


from copy import deepcopy
refprot = ev2a_prot
queryprot = cox2a_prot
gap_query = deepcopy( queryprot)
gap_query_seq = get_chain_seq( gap_query )[0]
print("Before deletion:", gap_query_seq )

alignment = model_seqAlign( refprot, queryprot,).visualize().map_matching_res( match_type = "exact", gaps = True)

print([ res.name if res != None else None for res in alignment.matched_res["Query"] ][:] )
print("Ref Len: {}\tQuery Len:{}".format(len(alignment.refseq), len(alignment.queryseq) ) )
print( "Query Macthed res Len: ", len(alignment.matched_res["Query"]))

print( "#############################################################")
gap_query = sele_pdb( gap_query, sele_res_idx, [  (10,20) ] )
gap_query_seq = get_chain_seq( gap_query )[0]
print("After deletion: ", gap_query_seq )


alignment = model_seqAlign( refprot, gap_query,).visualize().map_matching_res( match_type = "exact", gaps = True)

print([ res.name if res != None else None for res in alignment.matched_res["Query"] ][:] )
print("Ref Len: {}\tQuery Len:{}".format(len(alignment.refseq), len(alignment.queryseq) ) )
print( len(alignment.matched_res["Query"]))

Before deletion: GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
                |         |         |         |         *         |         |         |         |         +         |         |         |         |         *
Ref:   ------SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLD----EE
             |||:|||||:|||||||||.||.|.||||.:||||||:|||.||||||||.|.||||:|:|:.|||||||..|.|:.|:.|||||.|||||::||.|.|||||.||||||:|||:|:|:.||.|:||||||||||||:    |:
Query: GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
['SER', 'GLY', 'ALA', None, 'TYR', 'VAL', 'GLY', 'ASN', 'TYR', None, 'VAL', 'VAL', 'ASN', 'ARG', 'HIS', 'LEU', 'ALA', 'THR', 'HIS', None, 'ASP', 'TRP', None, 'ASN', None, 'VAL', 'TRP', 'GLU', 'ASP', N

In [20]:
from xaidar.data.molecModels import get_pdb_stats, get_chain_seq, get_res_CoM, get_chain_seq
from xaidar.data.molecModels import flatten_pdb
from xaidar.data.molecModels import sele_pdb, sele_AA, sele_closest_Chain, sele_model, sele_Lig
import gemmi
ev2a_pdb = gemmi.read_pdb("../../../tests/testdata/protein/ev2a.pdb")
cox2a_pdb = gemmi.read_pdb("../../../tests/testdata/protein/coxb4_2a.pdb")
ev2a1_pdb = gemmi.read_pdb("../../../tests/testdata/protein/ev2a_1.pdb")
# get_pdb_stats( ev2a_pdb)
# get_pdb_stats( cox2a_pdb)

ev2a_pdb = sele_pdb( ev2a_pdb, sele_model)
lig_pdb = sele_pdb( ev2a_pdb, sele_Lig)
lig = flatten_pdb( lig_pdb, level = "residue")
lig_CoM = get_res_CoM( lig)[0]
lig_CoM = gemmi.Position( *lig_CoM)
ev2a_pdb = sele_pdb( ev2a_pdb, sele_AA)
ev2a_pdb = sele_pdb( ev2a_pdb, sele_closest_Chain, lig_CoM)
# get_pdb_stats( ev2a_pdb)

cox2a_pdb = sele_pdb( cox2a_pdb, sele_AA)
cox2a_pdb = sele_pdb( cox2a_pdb, sele_model)
# get_pdb_stats( cox2a_pdb)



ev2a1_pdb = sele_pdb( ev2a1_pdb, sele_model)
lig_pdb = sele_pdb( ev2a1_pdb, sele_Lig)
lig = flatten_pdb( lig_pdb, level = "residue")
lig_CoM = get_res_CoM( lig)[0]
lig_CoM = gemmi.Position( *lig_CoM)
ev2a1_pdb = sele_pdb( ev2a1_pdb, sele_AA)
ev2a1_pdb = sele_pdb( ev2a1_pdb, sele_closest_Chain, lig_CoM)
get_pdb_stats( ev2a1_pdb)


ev2a_seq = get_chain_seq( flatten_pdb(ev2a_pdb, level = "chain") ) 
cox2a_seq = get_chain_seq( flatten_pdb(cox2a_pdb, level = "chain") ) 
ev2a1_seq = get_chain_seq( flatten_pdb(ev2a1_pdb, level = "chain") )
print(ev2a_seq)
print(cox2a_seq)
print(ev2a1_seq)



####################
Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.
['SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLDEE']
['GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ']
['SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLDEE']


In [21]:
# seqAlign(ev2a_seq[0], cox2a_seq[0][::-1], sanityCheck=True).align().visualize().find_orientation().visualize()

alignment = model_seqAlign( ev2a_pdb, cox2a_pdb).align().visualize().check_match()
print( alignment.matched_res)
print( alignment.matched_indices)

# print(len(alignment.matched_indices["Ref"]["Seq_Idx"]), alignment.matched_indices["Ref"]["Seq_Idx"] )
# print(len(alignment.matched_indices["Query"]["Seq_Idx"]), alignment.matched_indices["Query"]["Seq_Idx"])

                |         |         |         |         *         |         |         |         |         +         |         |         |         |         *
Ref:   ------SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLD----EE
             |||:|||||:|||||||||.||.|.||||.:||||||:|||.||||||||.|.||||:|:|:.|||||||..|.|:.|:.|||||.|||||::||.|.|||||.||||||:|||:|:|:.||.|:||||||||||||:    |:
Query: GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
Mismatch in residue serial numbers between reference and query.
{'Ref': [<gemmi.Residue 7(SER) with 6 atoms>, <gemmi.Residue 8(GLY) with 4 atoms>, <gemmi.Residue 9(ALA) with 5 atoms>, <gemmi.Residue 11(TYR) with 12 atoms>, <gemmi.Residue 12(VAL) with 7 atoms>, <gemmi.Residue 13(GLY) with 4 atoms>, <gemmi.Residue 14(ASN) with 8 atoms>, <gemmi.Residue 15(TYR) with 12 

In [ ]:
ev2a_res = flatten_pdb(ev2a_pdb, level = "residue")
cox2a_res = flatten_pdb(cox2a_pdb, level = "residue")

ev2a_match_idx =  alignment.matched_indices["Ref"]["Seq_Idx"]
cox2a_match_idx = alignment.matched_indices["Query"]["Seq_Idx"]

ev2a_match_res = [ ev2a_res[idx].seqid.num for idx in ev2a_match_idx ]
cox2a_match_res = [ cox2a_res[idx].seqid.num for idx in cox2a_match_idx ]

print( ev2a_match_res == cox2a_match_res )


False


In [9]:
alignment = model_seqAlign( ev2a_pdb, ev2a1_pdb).align().visualize().map_match_indices().test_match()
print( [ res.seqid.num for res in alignment.matched_res["Ref"] ] )
print( [ res.seqid.num for res in alignment.matched_res["Query"] ] )

                |         |         |         |         *         |         |         |         |         +         |         |         |         |
Ref:   SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLDEE
       ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Query: SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLDEE
Match of all residue serial numbers between reference and query.
[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 9

In [11]:
ev2a_res = flatten_pdb(ev2a_pdb, level = "residue")
ev2a1_res = flatten_pdb(ev2a1_pdb, level = "residue")

ev2a_match_idx =  alignment.matched_indices["Ref"]["Seq_Idx"]
ev2a1_match_idx = alignment.matched_indices["Query"]["Seq_Idx"]

ev2a_match_res = [ ev2a_res[idx].seqid.num for idx in ev2a_match_idx ]
ev2a1_match_res = [ ev2a1_res[idx].seqid.num for idx in ev2a1_match_idx ]

print( ev2a_match_res == ev2a1_match_res )
print(ev2a_match_res)
print(ev2a1_match_res)


True
[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146]
[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 

In [ ]:
res_seq_map = list(zip(ev2a_match_idx, ev2a1_match_idx)) # Pair of indicies in the sequences 
print(res_seq_map)
res_map = list(zip(ev2a_match_res, ev2a1_match_res) ) # Pair of residue numbers in the sequences
print(res_map)


[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9), (10, 10), (11, 11), (12, 12), (13, 13), (14, 14), (15, 15), (16, 16), (17, 17), (18, 18), (19, 19), (20, 20), (21, 21), (22, 22), (23, 23), (24, 24), (25, 25), (26, 26), (27, 27), (28, 28), (29, 29), (30, 30), (31, 31), (32, 32), (33, 33), (34, 34), (35, 35), (36, 36), (37, 37), (38, 38), (39, 39), (40, 40), (41, 41), (42, 42), (43, 43), (44, 44), (45, 45), (46, 46), (47, 47), (48, 48), (49, 49), (50, 50), (51, 51), (52, 52), (53, 53), (54, 54), (55, 55), (56, 56), (57, 57), (58, 58), (59, 59), (60, 60), (61, 61), (62, 62), (63, 63), (64, 64), (65, 65), (66, 66), (67, 67), (68, 68), (69, 69), (70, 70), (71, 71), (72, 72), (73, 73), (74, 74), (75, 75), (76, 76), (77, 77), (78, 78), (79, 79), (80, 80), (81, 81), (82, 82), (83, 83), (84, 84), (85, 85), (86, 86), (87, 87), (88, 88), (89, 89), (90, 90), (91, 91), (92, 92), (93, 93), (94, 94), (95, 95), (96, 96), (97, 97), (98, 98), (99, 99), (100, 100), (101, 1

In [30]:
test_res = ev2a_res[0]
test_res.name, test_res.seqid.num

('SER', 7)

# Extra Functions

- ## get_aa_distribution

In [19]:
# load test pdbs

import gemmi
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
cox2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/coxb4_2a.pdb")
ev2a1_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a_1.pdb")
from xaidar.data.protocols import protein_processing
from xaidar.data.molecModels import sele_pdb, sele_AA, sele_model
from xaidar.data.molecModels import get_pdb_stats
ev2a_prot = protein_processing(ev2a_pdb)
ev2a1_prot = protein_processing(ev2a1_pdb)
cox2a_prot = cox2a_pdb
for foo in [ sele_AA, sele_model]:
    cox2a_prot = sele_pdb(cox2a_prot, foo)
    
lst_prots = [ ev2a_prot, ev2a1_prot, cox2a_prot,  ]
ref_prot = ev2a_prot

In [ ]:
def get_aa_distribution( ref_model: gemmi.Structure, 
                         query_models_lst: list[gemmi.Structure],
                         ) -> dict:
    """
    Get the amino acid distribution at each position in the reference sequence
    across a list of query models.
    Args:
    - ref_model (gemmi.Structure): Reference protein structure.
    - query_models_lst (list[gemmi.Structure]): List of query protein structures.
    Returns:
    - dict: Dictionary with positions as keys and lists of amino acids as values.
    
    """
    ref_seq = get_chain_seq( ref_model )[0]
    aa_distrib_dict = {ref_aa_pos : [] for ref_aa_pos in range(len(ref_seq))}   # Dictionary to store AA distribution at each position in ref seq       
    for query_model in query_models_lst[:]:
        alignment  = model_seqAlign( ref_model, query_model)
        alignment.map_matching_res(match_type = "exact", gaps = True)
        for ref_idx in aa_distrib_dict.keys():                                       # Loop over each position in ref seq
            aa_distrib_dict[ref_idx].append( alignment.matched_res["Query"][ref_idx])
    return aa_distrib_dict




In [21]:
aa_distrib_dict = get_aa_distribution( ref_prot, lst_prots, )
print( [res.name if res != None else None for res in aa_distrib_dict[0] ] )
print([ [res.name if res != None else None for res in aa_distrib_dict[idx]] 
       for idx in aa_distrib_dict.keys() ] )

['SER', 'SER', 'SER']
[['SER', 'SER', 'SER'], ['GLY', 'GLY', 'GLY'], ['ALA', 'ALA', 'ALA'], ['ILE', 'ILE', None], ['TYR', 'TYR', 'TYR'], ['VAL', 'VAL', 'VAL'], ['GLY', 'GLY', 'GLY'], ['ASN', 'ASN', 'ASN'], ['TYR', 'TYR', 'TYR'], ['ARG', 'ARG', None], ['VAL', 'VAL', 'VAL'], ['VAL', 'VAL', 'VAL'], ['ASN', 'ASN', 'ASN'], ['ARG', 'ARG', 'ARG'], ['HIS', 'HIS', 'HIS'], ['LEU', 'LEU', 'LEU'], ['ALA', 'ALA', 'ALA'], ['THR', 'THR', 'THR'], ['HIS', 'HIS', 'HIS'], ['ASN', 'ASN', None], ['ASP', 'ASP', 'ASP'], ['TRP', 'TRP', 'TRP'], ['ALA', 'ALA', None], ['ASN', 'ASN', 'ASN'], ['LEU', 'LEU', None], ['VAL', 'VAL', 'VAL'], ['TRP', 'TRP', 'TRP'], ['GLU', 'GLU', 'GLU'], ['ASP', 'ASP', 'ASP'], ['SER', 'SER', None], ['SER', 'SER', None], ['ARG', 'ARG', 'ARG'], ['ASP', 'ASP', 'ASP'], ['LEU', 'LEU', 'LEU'], ['LEU', 'LEU', 'LEU'], ['VAL', 'VAL', 'VAL'], ['SER', 'SER', 'SER'], ['SER', 'SER', None], ['THR', 'THR', 'THR'], ['THR', 'THR', 'THR'], ['ALA', 'ALA', 'ALA'], ['GLN', 'GLN', None], ['GLY', 'GLY', 'GLY'